# Fraud Mitigation Agent · 08 Decision Engine

El resultado es una decisión explícita con reason codes y evidencia persistible.


In [ ]:
import os, sys, json
from pathlib import Path

REPO_DIR = globals().get("REPO_DIR", "/content/fraud-mitigation-agent-workshop")
src = Path(REPO_DIR) / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_mitigation_agent.synthetic import seed_demo_data
from fraud_mitigation_agent.local import InMemoryDB

if "db" not in globals():
    from fraud_mitigation_agent.config import Settings
    from fraud_mitigation_agent.db import get_client, get_database
    settings = Settings.from_env()
    if settings.mongodb_uri:
        client = get_client(settings.mongodb_uri)
        db = get_database(client, settings.database_name)
    else:
        db = InMemoryDB()
seed_demo_data(db, reset=False)
print("Runtime listo:", type(db).__name__)


In [ ]:
from fraud_mitigation_agent.agent import FraudAgent

agent = FraudAgent(db)
result = agent.analyze("tx-risky-001", persist=True)
print(json.dumps({k: v for k, v in result.items() if k != "trace"}, indent=2, default=str))
print("tool calls:", len(result["trace"]))
